In [1]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("..")

In [ ]:
import numpy as np
from config import ROBOTIQ_HANDE_GRIPPER

from meshgraphnet.simulator import Simulator

gripper = ROBOTIQ_HANDE_GRIPPER()

filepath = "../meshes/primitives/msh/Cuboid1_cg2.msh"
simulator = Simulator(filepath, std=0.01)

F = 20.0
z = 0.02

n = np.array([1.0, 0.0, 0.0]) * 1.0
p1 = np.array([0.015, 0.0, z])
f1 = np.array([0.0, 1.0, 0.0]) - n
p2 = np.array([-0.015, 0.0, z])
f2 = np.array([0.0, 1.0, 0.0]) + n

print("p1:", p1)
print("f1:", f1)
print("p2:", p2)
print("f2:", f2)

uh = simulator.run([(p1, f1)])
vm = simulator.compute_vm1(uh)
simulator.plot_vm(vm)
simulator.plot_vm_bottom(vm)

Info    : Reading '../meshes/primitives/msh/Cuboid1_cg2.msh'...
Info    : 27 entities
Info    : 21480 nodes
Info    : 16905 elements
Info    : Done reading '../meshes/primitives/msh/Cuboid1_cg2.msh'
Using Lagrange elements of order 2 for simulation.

p1: [0.015 0.    0.02 ]
f1: [-1.  0. 20.]
p2: [-0.015  0.     0.02 ]
f2: [ 1.  0. 20.]


Widget(value='<iframe src="http://localhost:36169/index.html?ui=P_0x73161fea8d40_24&reconnect=auto" class="pyv…

Widget(value='<iframe src="http://localhost:36169/index.html?ui=P_0x7315ee8976b0_25&reconnect=auto" class="pyv…

# Sample Grasps
Sample antipodal grasps on the object mesh.

In [ ]:
import sys

sys.path.append("..")

import meshio
from config import ROBOTIQ_HANDE_GRIPPER
from sampler import GraspSampler

from meshgraphnet.utils import msh_to_trimesh

msh = meshio.read("../meshes/test/msh/Bushing3_cg1.msh")
mesh = msh_to_trimesh(msh)
gripper = ROBOTIQ_HANDE_GRIPPER()
sampler = GraspSampler(mesh=mesh, gripper=gripper, mu=0.1)
grasps = sampler.sample(n_samples=10, debug=False)


# Find Optimal Grasps
Load GNN model and select the best grasp (with highest score) according to the model's predictions.

In [ ]:
import sys

sys.path.append("..")

import meshio
import torch
from config import ROBOTIQ_HANDE_GRIPPER
from optimizer import GNNBasedGraspOptimizer

from meshgraphnet.nets import EncodeProcessDecode
from meshgraphnet.normalizer import Normalizer

gripper = ROBOTIQ_HANDE_GRIPPER()

msh = meshio.read("../meshes/test/msh/L-Bracket4_cg1.msh")

checkpoint = torch.load(
    "../models/Model0.pth",
    map_location=torch.device("cpu"),
    weights_only=True,
)
model_state_dict = checkpoint["model_state_dict"]
params = checkpoint["params"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
normalizer = Normalizer(
    num_features=params["node_dim"],
    num_categorical=params["num_categorical"],
    device=device,
    stats=checkpoint["stats"],
)
model = EncodeProcessDecode(
    node_dim=params["node_dim"],
    edge_dim=params["edge_dim"],
    output_dim=params["output_dim"],
    latent_dim=params["latent_dim"],
    message_passing_steps=params["message_passing_steps"],
    use_layer_norm=params["use_layer_norm"],
).to(device)
model.load_state_dict(model_state_dict)
model.eval()

optimizer = GNNBasedGraspOptimizer(gripper, model, normalizer, device=device)
grasps = optimizer.optimize(msh, mu=0.1, k=20)


# Visualize Grasps
Visualize the optimal grasps on the object mesh.

In [ ]:
from sampler import GraspSampler

from meshgraphnet.utils import msh_to_trimesh

mesh = msh_to_trimesh(msh)

score, grasp = grasps[5]
print("Best grasp score:", score)
print("Best grasp:", grasp)

sampler = GraspSampler(mesh=mesh, gripper=gripper, mu=0.1)
scene = sampler.visualize_grasp(grasp)

In [ ]:
import trimesh
from scipy.spatial.transform import Rotation as R

base_pose = grasp.pose
print("Base:")
print("   Position:", base_pose.pos)
print("   Orientation (quaternion):", base_pose.quat)

fingertip_pose = base_pose.se3() @ gripper.tf_base_to_fingertip()
print("Fingertip:")
print("   Position:", fingertip_pose[:3, 3])
print("   Orientation (quaternion):", R.from_matrix(fingertip_pose[:3, :3]).as_quat())

x_dir = fingertip_pose[:3, 0]

# Find the contact points on the object mesh
intersector = trimesh.ray.ray_triangle.RayMeshIntersector(mesh)
locs1, index_ray1, index_tri1 = intersector.intersects_location(
    [fingertip_pose[:3, 3]], [x_dir], multiple_hits=True
)
locs2, index_ray2, index_tri2 = intersector.intersects_location(
    [fingertip_pose[:3, 3]], [-x_dir], multiple_hits=True
)

scene = trimesh.Scene()
scene.add_geometry(mesh, node_name="object")

base_visual = trimesh.primitives.Sphere(radius=0.005, center=base_pose.pos)
base_visual.visual.face_colors = [132, 32, 55, 255]
scene.add_geometry(base_visual, node_name="base")

fingertip_visual = trimesh.primitives.Sphere(radius=0.005, center=fingertip_pose[:3, 3])
fingertip_visual.visual.face_colors = [132, 32, 55, 255]
scene.add_geometry(fingertip_visual, node_name="fingertip")

contact_visual = trimesh.primitives.Sphere(radius=0.005, center=locs1[0])
contact_visual.visual.face_colors = [32, 132, 55, 255]
scene.add_geometry(contact_visual, node_name="contact")

contact_visual2 = trimesh.primitives.Sphere(radius=0.005, center=locs2[0])
contact_visual2.visual.face_colors = [32, 132, 55, 255]
scene.add_geometry(contact_visual2, node_name="contact2")

scene.show(viewer="gl")

In [ ]:
from config import ROBOTIQ_HANDE_GRIPPER

gripper = ROBOTIQ_HANDE_GRIPPER()
gripper.show_box_fingers(0.025)